# 面向真实用例的模型选择实用指南

## 目的与受众

本 cookbook 是一份实用指南，帮助你为特定工作负载选择合适的 OpenAI 模型（GPT 4.1、o3 和 o4-mini 之间）、编写提示并进行部署。我们不提供详尽的文档，而是提供可落地的决策框架和真实示例，帮助解决方案工程师、技术客户经理、合作伙伴架构师以及具备一定技术背景的从业者快速构建可用的解决方案。内容聚焦于当前的模型能力、面向垂直行业的实现以及当下的行业需求，并提供从模型选择到生产部署的清晰路径。每个章节都提供简洁、可调整的代码示例，你可以立即将其应用到自己的用例中，同时也指向现有资源，方便你深入了解特定主题。

> 注意：以下规范性指导和实验是使用当前可用的最新 SOTA 模型进行的。考虑到不同的场景和时间线，这些指标将来必然会发生变化。

## 如何使用本 Cookbook

本 cookbook 分为若干独立章节，帮助你快速找到所需信息。每个章节涵盖模型选择、实现和部署的一个特定方面。

1. **[目的与受众](#目的与受众)**：概述本 cookbook 面向哪些读者以及涵盖哪些内容。
2. **[模型指南](#模型指南)**：帮助你根据需求选择合适模型的快速参考，包括基于不同用例场景映射的模型对比和演进图。
3. **用例**：
   - **[3A. 面向法律问答的长上下文 RAG](#3a-用例面向法律问答的长上下文-rag)**：构建一个智能体系统，回答来自复杂法律文档的问题。
   - **[3B. 面向制药研发的 AI 联合科学家](#3b-用例面向制药研发的-ai-联合科学家)**：利用多智能体系统加速制药研究中的实验设计。
   - **[3C. 保险理赔处理](#3c-用例保险理赔处理)**：利用视觉和推理能力对手写保险表单进行数字化和校验。
4. **[从原型到生产](#prototype-to-production)**：帮助你从原型过渡到生产的检查清单。
5. **[适配决策树](#适配决策树)**：根据具体需求指导模型选择的流程图。
6. **[附录](#附录)**：参考资料，包括定价、延迟、提示模式以及外部资源链接。

如需快速决策，请重点阅读“模型指南”和“适配决策树”两个章节。如需了解实现细节，请查看与你需求相关的具体用例。


================================================================================



## 模型指南

## 2.1 模型简介矩阵

| 模型 | 核心优势 | 首选场景 | 注意事项 | 升级 / 降级路径 |
| :---- | :---- | :---- | :---- | :---- |
| GPT‑4o | 实时语音 / 视觉对话 | 实时多模态智能体 | 在文本 SOTA（state-of-the-art，当前最佳水平）上略逊于 4.1 | 需要深度推理 → o4‑mini |
| GPT‑4.1 | 100 万 token 文本准确率之王 | 长文档分析、代码审查 | 不能原生推理；成本高于 mini 系列 | 预算紧张 → 4.1‑mini / nano |
| o3 | 深度使用工具的智能体 | 高风险、多步骤推理 | 延迟和价格 | 成本/延迟 → o4‑mini |
| o4‑mini | 便宜、快速的推理 | 大批量的“足够好”逻辑 | 与 o3 相比深度有上限 | 准确率至关重要 → o3 |

*（完整的价格与用途表 → [6.1 节](#附录)）*

## 2.2 模型演进一览

OpenAI 的模型阵容不断演进，以满足不同维度上的专门需求。这些图展示了当前的模型家族及其相互关系。

### 根本区别：“o 系列”与“GPT”模型

OpenAI 提供两个不同的模型家族，各有独特优势：

- **GPT 模型（4o、4.1）**：针对通用任务优化，指令遵循能力出色。GPT-4.1 擅长处理长上下文（100 万 token），而 GPT-4o 有面向实时语音、文本转语音和语音转文本的变体。GPT-4.1 还有 mini 和 nano 变体，GPT-4o 则有 mini 变体。这些变体比完整版更便宜、更快。

- **o 系列模型（o3、o4-mini）**：专门用于深度推理和逐步解决问题。这些模型擅长需要逻辑思维和工具使用的复杂多阶段任务。当准确率和推理深度至关重要时，请选择它们。这些模型还有一个可选的 `reasoning_effort` 参数（可设置为 `low`、`medium` 或 `high`），让用户可以控制用于推理的 token 数量。

### OpenAI 模型演进 

![OpenAI 模型演进](../../../images/2.2_model_evolution.png)

### 关键特性

- **GPT-4.1 家族**：针对长上下文处理优化，具有 100 万 token 的上下文窗口。
- **o3**：专门用于深度多步骤推理。 
- **o4-mini**：以更低的成本结合了推理能力和视觉能力。

每个模型都在不同场景中表现出色，其优势互补，可以组合用于复杂的工作流。

在本 cookbook 中，我们只对 GPT-4.1 系列模型、o3 和 o4-mini 进行了实验，没有对 GPT-4o 系列模型进行实验。

================================================================================



## 3A. 用例：面向法律问答的长上下文 RAG

![面向法律问答的长上下文 RAG](../../../images/3A_rag_task_card.png)
## 🗂️ 太长不看矩阵

下表总结了**这一特定长上下文智能体式 RAG 实现**的核心技术选择及其理由。

| 层 | 选择 | 用途 |
| :---- | :---- | :---- |
| **分块** | 感知句子边界的切分器 | 将文档切分为 20 个大小相等的块，并遵循句子边界。 |
| **路由** | `gpt-4.1-mini` | 利用自然语言理解识别相关块，无需嵌入索引。 |
| **路径选择** | `select(ids=[...])` 和 `scratchpad(text="...")` | 在逐层深入文档层级的过程中记录推理过程。 |
| **引用** | 段落级 | 在精确度与成本之间取得平衡；为答案提供有意义的上下文。 |
| **综合** | `gpt-4.1`（Structured Output） | 直接根据选中的段落生成带引用的答案。 |
| **验证** | `o4-mini`（LLM-as-Judge） | 校验事实准确性和引用的正确性。 |

*注意：价格和模型标识符截至 2025 年 4 月准确，可能会发生变化。*

本节概述了一个检索增强生成（RAG）系统的构建过程，该系统旨在准确回答有关复杂冗长的程序性文本的问题，并以*《商标审判和上诉委员会程序手册》（Trademark Trial and Appeal Board Manual of Procedure，TBMP）*作为代表性案例。TBMP 是一份重要的法律资源，详细说明了在 USPTO 商标审判和上诉委员会进行商标诉讼的程序，知识产权律师和法律专业人士经常查阅。通过利用最新的 OpenAI 模型，该系统借助先进的语言理解和动态检索能力，增强了对晦涩法律内容的理解和可解释性，能够给出精确且结合上下文的回答。

这些方法也可以应用于其他需要从复杂文档中精确检索信息的用例，例如医疗合规手册、金融监管框架或技术文档系统，在这些场景中，准确性、引用和可审计性都是关键要求。

## 1\. 场景速览

* **语料库：** 主要文档是[《商标审判和上诉委员会程序手册》（TBMP，2024 版）](https://www.uspto.gov/sites/default/files/documents/tbmp-Master-June2024.pdf)。该手册包含详细的程序规则和指南，共 1194 页。  
* **用户：** 目标用户是知识产权（IP）诉讼律师助理和律师助手，他们需要*仅*基于 TBMP 快速、准确地回答程序性问题。  
* **典型问题：** 用户提出的问题需要综合信息并给出引用，例如：  
  1. "What are the requirements for filing a motion to compel discovery according to the TBMP?"（根据 TBMP，提交强制披露动议有哪些要求？）  
  2. "What deadlines apply to discovery conferences as specified in the manual?"（手册中规定的证据开示会议适用哪些截止期限？）  
  3. "Explain how the Board handles claims of attorney-client privilege during depositions according to the TBMP."（根据 TBMP，解释委员会在证词录取过程中如何处理律师-委托人特权主张。）  
  4. "Enumerate the Fed. R. Civ. P. 11 sanctions the Board can invoke according to the TBMP."（根据 TBMP，列举委员会可援引的 Fed. R. Civ. P. 11 制裁措施。）  

*注意：根据你的具体部署环境，你可能需要调整部分实现步骤以满足基础设施要求。*

> 虽然 OpenAI 的 File Search 工具为许多用例提供了一个不错的起点，但本节介绍一种不同的方法：利用百万 token 的上下文窗口来处理大型文档，无需任何预处理或向量数据库。这里描述的智能体式方法可以实现零延迟摄取、动态的检索粒度以及细粒度的引用可追溯性。

## 2\. 智能体式 RAG 流程

在深入实现之前，我们先了解整体方法：

1. **将整个文档加载**到上下文窗口中
2. **切分为 20 个块**，并遵循句子边界
3. **询问模型**哪些块可能包含相关信息
4. **逐层深入**选中的块，将其进一步切分
5. **重复**直到达到段落级内容
6. 根据选中的段落**生成答案**
7. **验证答案**的事实准确性

这种分层导航方法模拟了人类浏览文档的方式：先略读，再聚焦相关章节，然后是具体小节，最后只阅读最相关的段落。

![分层路由器](../../../images/3A_rag_hierarchical_router.png)


## 智能体式 RAG 系统：模型使用情况

| 处理阶段 | 使用的模型 | 用途 |
|---------------|------------|---------|
| 初始路由 | `gpt-4.1-mini` | 识别哪些文档块可能包含相关信息 |
| 分层导航 | `gpt-4.1-mini` | 继续逐层深入，找到最相关的段落 |
| 答案生成 | `gpt-4.1` | 根据选中的段落生成带引用的结构化回答 |
| 答案验证 | `o4-mini` | 校验事实准确性和引用的正确使用 |

这种零预处理的方法利用大上下文窗口即时导航文档，模拟人类浏览文档以查找相关信息的方式。 

## 3\. 实现

我们一步步实现这种方法。

首先安装所需的包。

In [1]:
%pip install tiktoken pypdf nltk openai pydantic --quiet

Note: you may need to restart the kernel to use updated packages.


### 3.1 加载文档

首先，我们加载文档并检查其大小。在本指南中，我们将聚焦于第 100-900 节，这些章节涵盖了从核心程序到 Review of Decision of Board 的内容。第 1000 节及之后的内容（Interferences、Concurrent Use Proceedings、Ex Parte Appeals）属于专门程序，不在当前范围之内。

In [3]:
import requests
from io import BytesIO
from pypdf import PdfReader
import re
import tiktoken
from nltk.tokenize import sent_tokenize
import nltk
from typing import List, Dict, Any

# Download nltk data if not already present
nltk.download('punkt_tab')

def load_document(url: str) -> str:
    """Load a document from a URL and return its text content."""
    print(f"Downloading document from {url}...")
    response = requests.get(url)
    response.raise_for_status()
    pdf_bytes = BytesIO(response.content)
    pdf_reader = PdfReader(pdf_bytes)
    
    full_text = ""
    

    max_page = 920  # Page cutoff before section 1000 (Interferences)
    for i, page in enumerate(pdf_reader.pages):
        if i >= max_page:
            break
        full_text += page.extract_text() + "\n"
    
    # Count words and tokens
    word_count = len(re.findall(r'\b\w+\b', full_text))
    
    tokenizer = tiktoken.get_encoding("o200k_base")
    token_count = len(tokenizer.encode(full_text))
    
    print(f"Document loaded: {len(pdf_reader.pages)} pages, {word_count} words, {token_count} tokens")
    return full_text

# Load the document
tbmp_url = "https://www.uspto.gov/sites/default/files/documents/tbmp-Master-June2024.pdf"
document_text = load_document(tbmp_url)

# Show the first 500 characters
print("\nDocument preview (first 500 chars):")
print("-" * 50)
print(document_text[:500])
print("-" * 50)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kmurali/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Document loaded: 1194 pages, 595197 words, 932964 tokens

Document preview (first 500 chars):
--------------------------------------------------
TRADEMARK TRIAL AND
APPEAL BOARD MANUAL
OF PROCEDURE (TBMP)
 June 2024
June   2024
United States Patent and Trademark Office
PREFACE TO THE JUNE 2024 REVISION
The June 2024 revision of the Trademark Trial and Appeal Board Manual of Procedure is an update of the
June 2023 edition. This update is moderate in nature and incorporates relevant case law issued between March
3, 2023 and March 1, 2024.
The title of the manual is abbreviated as “TBMP.” A citation to a section of the manual may be written
--------------------------------------------------


可以看到，该文档超过 90 万个 token！虽然 GPT 4.1 的上下文长度能够容纳它，但我们还希望获得可验证的引用，因此将采用递归分块策略。

### 3.2 带最小 token 数的改进版 20 块切分器

现在，我们来创建一个改进的函数，将文档切分为 20 个块，确保每个块都达到最小 token 数，并遵循句子边界。

> 20 是针对这个特定文档/任务凭经验选定的数字，对于其他文档，可能需要根据大小和结构进行调整（数字越大，块越细）。不过这里的关键原则是将文档的各个部分切分开，让语言模型来判断哪些部分相关。同样的思路也适用于本 cookbook 后面将介绍的 `max_depth` 参数。

In [4]:
# Global tokenizer name to use consistently throughout the code
TOKENIZER_NAME = "o200k_base"

def split_into_20_chunks(text: str, min_tokens: int = 500) -> List[Dict[str, Any]]:
    """
    Split text into up to 20 chunks, respecting sentence boundaries and ensuring
    each chunk has at least min_tokens (unless it's the last chunk).
    
    Args:
        text: The text to split
        min_tokens: The minimum number of tokens per chunk (default: 500)
    
    Returns:
        A list of dictionaries where each dictionary has:
        - id: The chunk ID (0-19)
        - text: The chunk text content
    """
    # First, split the text into sentences
    sentences = sent_tokenize(text)
    
    # Get tokenizer for counting tokens
    tokenizer = tiktoken.get_encoding(TOKENIZER_NAME)
    
    # Create chunks that respect sentence boundaries and minimum token count
    chunks = []
    current_chunk_sentences = []
    current_chunk_tokens = 0
    
    for sentence in sentences:
        # Count tokens in this sentence
        sentence_tokens = len(tokenizer.encode(sentence))
        
        # If adding this sentence would make the chunk too large AND we already have the minimum tokens,
        # finalize the current chunk and start a new one
        if (current_chunk_tokens + sentence_tokens > min_tokens * 2) and current_chunk_tokens >= min_tokens:
            chunk_text = " ".join(current_chunk_sentences)
            chunks.append({
                "id": len(chunks),  # Integer ID instead of string
                "text": chunk_text
            })
            current_chunk_sentences = [sentence]
            current_chunk_tokens = sentence_tokens
        else:
            # Add this sentence to the current chunk
            current_chunk_sentences.append(sentence)
            current_chunk_tokens += sentence_tokens
    
    # Add the last chunk if there's anything left
    if current_chunk_sentences:
        chunk_text = " ".join(current_chunk_sentences)
        chunks.append({
            "id": len(chunks),  # Integer ID instead of string
            "text": chunk_text
        })
    
    # If we have more than 20 chunks, consolidate them
    if len(chunks) > 20:
        # Recombine all text
        all_text = " ".join(chunk["text"] for chunk in chunks)
        # Re-split into exactly 20 chunks, without minimum token requirement
        sentences = sent_tokenize(all_text)
        sentences_per_chunk = len(sentences) // 20 + (1 if len(sentences) % 20 > 0 else 0)
        
        chunks = []
        for i in range(0, len(sentences), sentences_per_chunk):
            # Get the sentences for this chunk
            chunk_sentences = sentences[i:i+sentences_per_chunk]
            # Join the sentences into a single text
            chunk_text = " ".join(chunk_sentences)
            # Create a chunk object with ID and text
            chunks.append({
                "id": len(chunks),  # Integer ID instead of string
                "text": chunk_text
            })
    
    # Print chunk statistics
    print(f"Split document into {len(chunks)} chunks")
    for i, chunk in enumerate(chunks):
        token_count = len(tokenizer.encode(chunk["text"]))
        print(f"Chunk {i}: {token_count} tokens")
    
    return chunks

# Split the document into 20 chunks with minimum token size
document_chunks = split_into_20_chunks(document_text, min_tokens=500)

Split document into 20 chunks
Chunk 0: 42326 tokens
Chunk 1: 42093 tokens
Chunk 2: 42107 tokens
Chunk 3: 39797 tokens
Chunk 4: 58959 tokens
Chunk 5: 48805 tokens
Chunk 6: 37243 tokens
Chunk 7: 33453 tokens
Chunk 8: 38644 tokens
Chunk 9: 49402 tokens
Chunk 10: 51568 tokens
Chunk 11: 49586 tokens
Chunk 12: 47722 tokens
Chunk 13: 48952 tokens
Chunk 14: 44994 tokens
Chunk 15: 50286 tokens
Chunk 16: 54424 tokens
Chunk 17: 62651 tokens
Chunk 18: 47430 tokens
Chunk 19: 42507 tokens


### 3.3 带改进工具 schema 的路由函数

现在，我们来创建路由函数，它将选择相关块并维护一个草稿本（scratchpad）。

> 维护草稿本可以让模型随时间跟踪决策标准和推理过程。该实现使用 GPT-4.1-mini 进行两轮处理：首先要求模型通过工具调用更新草稿本（tool_choice="required"），然后请求以结构化 JSON 输出块的选择结果。这种方法能更清楚地展示模型的推理过程，同时确保为下游处理提供一致的结构化输出。

In [5]:
from openai import OpenAI
import json
from typing import List, Dict, Any

# Initialize OpenAI client
client = OpenAI()

def route_chunks(question: str, chunks: List[Dict[str, Any]], 
                depth: int, scratchpad: str = "") -> Dict[str, Any]:
    """
    Ask the model which chunks contain information relevant to the question.
    Maintains a scratchpad for the model's reasoning.
    Uses structured output for chunk selection and required tool calls for scratchpad.
    
    Args:
        question: The user's question
        chunks: List of chunks to evaluate
        depth: Current depth in the navigation hierarchy
        scratchpad: Current scratchpad content
    
    Returns:
        Dictionary with selected IDs and updated scratchpad
    """
    print(f"\n==== ROUTING AT DEPTH {depth} ====")
    print(f"Evaluating {len(chunks)} chunks for relevance")
    
    # Build system message
    system_message = """You are an expert document navigator. Your task is to:
1. Identify which text chunks might contain information to answer the user's question
2. Record your reasoning in a scratchpad for later reference
3. Choose chunks that are most likely relevant. Be selective, but thorough. Choose as many chunks as you need to answer the question, but avoid selecting too many.

First think carefully about what information would help answer the question, then evaluate each chunk.
"""

    # Build user message with chunks and current scratchpad
    user_message = f"QUESTION: {question}\n\n"
    
    if scratchpad:
        user_message += f"CURRENT SCRATCHPAD:\n{scratchpad}\n\n"
    
    user_message += "TEXT CHUNKS:\n\n"
    
    # Add each chunk to the message
    for chunk in chunks:
        user_message += f"CHUNK {chunk['id']}:\n{chunk['text']}\n\n"
    
    # Define function schema for scratchpad tool calling
    tools = [
        {
            "type": "function",
            "name": "update_scratchpad",
            "description": "Record your reasoning about why certain chunks were selected",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "Your reasoning about the chunk(s) selection"
                    }
                },
                "required": ["text"],
                "additionalProperties": False
            }
        }
    ]
    
    # Define JSON schema for structured output (selected chunks)
    text_format = {
        "format": {
            "type": "json_schema",
            "name": "selected_chunks",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "chunk_ids": {
                        "type": "array",
                        "items": {"type": "integer"},
                        "description": "IDs of the selected chunks that contain information to answer the question"
                    }
                },
                "required": [
                    "chunk_ids"
                ],
                "additionalProperties": False
            }
        }
    }
    
    # First pass: Call the model to update scratchpad (required tool call)
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message + "\n\nFirst, you must use the update_scratchpad function to record your reasoning."}
    ]
    
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=messages,
        tools=tools,
        tool_choice="required"
    )
    
    # Process the scratchpad tool call
    new_scratchpad = scratchpad
    
    for tool_call in response.output:
        if tool_call.type == "function_call" and tool_call.name == "update_scratchpad":
            args = json.loads(tool_call.arguments)
            scratchpad_entry = f"DEPTH {depth} REASONING:\n{args.get('text', '')}"
            if new_scratchpad:
                new_scratchpad += "\n\n" + scratchpad_entry
            else:
                new_scratchpad = scratchpad_entry
            
            # Add function call and result to messages
            messages.append(tool_call)
            messages.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": "Scratchpad updated successfully."
            })
    
    # Second pass: Get structured output for chunk selection
    messages.append({"role": "user", "content": "Now, select the chunks that could contain information to answer the question. Return a JSON object with the list of chunk IDs."})
    
    response_chunks = client.responses.create(
        model="gpt-4.1-mini",
        input=messages,
        text=text_format
    )
    
    # Extract selected chunk IDs from structured output
    selected_ids = []
    if response_chunks.output_text:
        try:
            # The output_text should already be in JSON format due to the schema
            chunk_data = json.loads(response_chunks.output_text)
            selected_ids = chunk_data.get("chunk_ids", [])
        except json.JSONDecodeError:
            print("Warning: Could not parse structured output as JSON")
    
    # Display results
    print(f"Selected chunks: {', '.join(str(id) for id in selected_ids)}")
    print(f"Updated scratchpad:\n{new_scratchpad}")
    
    return {
        "selected_ids": selected_ids,
        "scratchpad": new_scratchpad
    }

### 3.4 递归导航函数

现在，我们来创建逐层深入文档的递归导航函数。`max_depth` 是向下深入的最大层数（同时要考虑 token 下限）：

In [6]:
def navigate_to_paragraphs(document_text: str, question: str, max_depth: int = 1) -> Dict[str, Any]:
    """
    Navigate through the document hierarchy to find relevant paragraphs.
    
    Args:
        document_text: The full document text
        question: The user's question
        max_depth: Maximum depth to navigate before returning paragraphs (default: 1)
    
    Returns:
        Dictionary with selected paragraphs and final scratchpad
    """
    scratchpad = ""
    
    # Get initial chunks with min 500 tokens
    chunks = split_into_20_chunks(document_text, min_tokens=500)
    
    # Navigator state - track chunk paths to maintain hierarchy
    chunk_paths = {}  # Maps numeric IDs to path strings for display
    for chunk in chunks:
        chunk_paths[chunk["id"]] = str(chunk["id"])
    
    # Navigate through levels until max_depth or until no chunks remain
    for current_depth in range(max_depth + 1):
        # Call router to get relevant chunks
        result = route_chunks(question, chunks, current_depth, scratchpad)
        
        # Update scratchpad
        scratchpad = result["scratchpad"]
        
        # Get selected chunks
        selected_ids = result["selected_ids"]
        selected_chunks = [c for c in chunks if c["id"] in selected_ids]
        
        # If no chunks were selected, return empty result
        if not selected_chunks:
            print("\nNo relevant chunks found.")
            return {"paragraphs": [], "scratchpad": scratchpad}
        
        # If we've reached max_depth, return the selected chunks
        if current_depth == max_depth:
            print(f"\nReturning {len(selected_chunks)} relevant chunks at depth {current_depth}")
            
            # Update display IDs to show hierarchy
            for chunk in selected_chunks:
                chunk["display_id"] = chunk_paths[chunk["id"]]
                
            return {"paragraphs": selected_chunks, "scratchpad": scratchpad}
        
        # Prepare next level by splitting selected chunks further
        next_level_chunks = []
        next_chunk_id = 0  # Counter for new chunks
        
        for chunk in selected_chunks:
            # Split this chunk into smaller pieces
            sub_chunks = split_into_20_chunks(chunk["text"], min_tokens=200)
            
            # Update IDs and maintain path mapping
            for sub_chunk in sub_chunks:
                path = f"{chunk_paths[chunk['id']]}.{sub_chunk['id']}"
                sub_chunk["id"] = next_chunk_id
                chunk_paths[next_chunk_id] = path
                next_level_chunks.append(sub_chunk)
                next_chunk_id += 1
        
        # Update chunks for next iteration
        chunks = next_level_chunks

### 3.5 针对示例问题运行改进后的导航

我们用改进后的方法针对一个示例问题运行导航：

In [7]:
# Run the navigation for a sample question
question = "What format should a motion to compel discovery be filed in? How should signatures be handled?"
navigation_result = navigate_to_paragraphs(document_text, question, max_depth=2)

# Sample retrieved paragraph
print("\n==== FIRST 3 RETRIEVED PARAGRAPHS ====")
for i, paragraph in enumerate(navigation_result["paragraphs"][:3]):
    display_id = paragraph.get("display_id", str(paragraph["id"]))
    print(f"\nPARAGRAPH {i+1} (ID: {display_id}):")
    print("-" * 40)
    print(paragraph["text"])
    print("-" * 40)

Split document into 20 chunks
Chunk 0: 42326 tokens
Chunk 1: 42093 tokens
Chunk 2: 42107 tokens
Chunk 3: 39797 tokens
Chunk 4: 58959 tokens
Chunk 5: 48805 tokens
Chunk 6: 37243 tokens
Chunk 7: 33453 tokens
Chunk 8: 38644 tokens
Chunk 9: 49402 tokens
Chunk 10: 51568 tokens
Chunk 11: 49586 tokens
Chunk 12: 47722 tokens
Chunk 13: 48952 tokens
Chunk 14: 44994 tokens
Chunk 15: 50286 tokens
Chunk 16: 54424 tokens
Chunk 17: 62651 tokens
Chunk 18: 47430 tokens
Chunk 19: 42507 tokens

==== ROUTING AT DEPTH 0 ====
Evaluating 20 chunks for relevance
Selected chunks: 0, 1, 2, 3, 4, 5, 6, 7, 8
Updated scratchpad:
DEPTH 0 REASONING:
The user wants to know the format requirements for filing a motion to compel discovery and how signatures should be handled for such motions. 

Based on the evaluation of chunks:
- Chunks 0, 1, 2, 3, 4, 5, 6, 7, 8 are highly relevant since they cover general requirements for submissions, motions, signatures, service, and specifically for motions and discovery in TTAB pro

GPT 4.1-mini 的结果展示了对文档中相关部分的迭代提取，而草稿本则解释了它在这一过程中的思考！在第 1 层，模型识别出 "*Detailed rules for signatures on submissions including motions*" 和 "*use of ESTTA, required signature format including electronic signatures with the symbol method '/sig/'*" 是回答该查询所需的关键部分。

到第 2 层时，草稿本展现出了老练的判断力：它精确地分离出包含电子签名关键规定的块（第 5-12 块），同时意识到哪些内容缺失，并指出 "*discovery-related motions... should be in chunks from 400 onwards (although these aren't fully visible here...)*"。

这一过程展示了 GPT 4.1 如何像法律分析师一样工作：迭代地深入挖掘相关内容，并在此过程中解释其推理（这让调试模型*为什么*选择这些块变得更容易）

### 3.6 答案生成

现在，我们使用 GPT-4.1 根据检索到的段落生成答案。 

> 这里我们用了一个巧妙的技巧：动态构建一个 Literal 列表（这会强制模型的答案只能是我们提供的选项之一——在本例中即段落 ID）。我们能提供的选项数量有一定限制，因此如果你发现系统引用了超过 500 个文档，这个方案可能就不适用了。在这种情况下，你可以加一个过滤器，将潜在引用控制在 500 个以内；或者让模型在回答中引用确切的 ID，然后对回答进行后处理以提取 ID，从而得到引用（例如它可能会写 "... [doc 0.0.12]"，你可以用正则表达式提取该引用）。



In [8]:
from typing import List, Dict, Any
from pydantic import BaseModel, field_validator

class LegalAnswer(BaseModel):
    """Structured response format for legal questions"""
    answer: str
    citations: List[str]
    
    @field_validator('citations')
    def validate_citations(cls, citations, info):
        # Access valid_citations from the model_config
        valid_citations = info.data.get('_valid_citations', [])
        if valid_citations:
            for citation in citations:
                if citation not in valid_citations:
                    raise ValueError(f"Invalid citation: {citation}. Must be one of: {valid_citations}")
        return citations

def generate_answer(question: str, paragraphs: List[Dict[str, Any]], 
                   scratchpad: str) -> LegalAnswer:
    """Generate an answer from the retrieved paragraphs."""
    print("\n==== GENERATING ANSWER ====")
    
    # Extract valid citation IDs
    valid_citations = [str(p.get("display_id", str(p["id"]))) for p in paragraphs]
    
    if not paragraphs:
        return LegalAnswer(
            answer="I couldn't find relevant information to answer this question in the document.",
            citations=[],
            _valid_citations=[]
        )
    
    # Prepare context for the model
    context = ""
    for paragraph in paragraphs:
        display_id = paragraph.get("display_id", str(paragraph["id"]))
        context += f"PARAGRAPH {display_id}:\n{paragraph['text']}\n\n"
    
    system_prompt = """You are a legal research assistant answering questions about the 
Trademark Trial and Appeal Board Manual of Procedure (TBMP).

Answer questions based ONLY on the provided paragraphs. Do not rely on any foundation knowledge or external information or extrapolate from the paragraphs.
Cite phrases of the paragraphs that are relevant to the answer. This will help you be more specific and accurate.
Include citations to paragraph IDs for every statement in your answer. Valid citation IDs are: {valid_citations_str}
Keep your answer clear, precise, and professional.
"""
    valid_citations_str = ", ".join(valid_citations)
    
    # Call the model using structured output
    response = client.responses.parse(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": system_prompt.format(valid_citations_str=valid_citations_str)},
            {"role": "user", "content": f"QUESTION: {question}\n\nSCRATCHPAD (Navigation reasoning):\n{scratchpad}\n\nPARAGRAPHS:\n{context}"}
        ],
        text_format=LegalAnswer,
        temperature=0.3
    )
    
    # Add validation information after parsing
    response.output_parsed._valid_citations = valid_citations
    
    print(f"\nAnswer: {response.output_parsed.answer}")
    print(f"Citations: {response.output_parsed.citations}")

    return response.output_parsed

# Generate an answer
answer = generate_answer(question, navigation_result["paragraphs"], 
                       navigation_result["scratchpad"])


==== GENERATING ANSWER ====

Answer: A motion to compel discovery must be filed electronically with the Trademark Trial and Appeal Board (TTAB) through ESTTA, unless ESTTA is unavailable due to technical problems or there are extraordinary circumstances, in which case a paper submission may be permitted with a written explanation ("Documents that relate to proceedings before the Trademark Trial and Appeal Board must be filed electronically with the Board through ESTTA"; "The rules require that all submissions must be made to the Board electronically, currently through ESTTA, subject to certain limited exceptions permitting submissions to be made on paper. Any permitted paper submission must be accompanied by a written explanation showing that ESTTA was unavailable due to technical problems, or that extraordinary circumstances are present, and, where required, a Petition to the Director with the requisite petition fee" 0.0.5.0, 0.0.5.5.7.3).

The motion should include a title describin

GPT 4.1 在整个回答中有效地融入了引用，同时保持了清晰的信息流。每项程序要求都关联到具体的权威参考（如 "0.0.5.0" 和 "0.0.5.5.6.2"），使回答既信息丰富，又来源精确。 

它没有简单地在末尾列出引用，而是在每项关键要求之后用括号标注，将引用直接融入内容之中。这种方式将对规则的常规复述转变为论据充分的法律分析：关于 ESTTA 提交程序、电子签名要求和纸质提交例外情况的陈述，都会立即有相应的法规引用作为支撑。

### 3.7 答案验证

我们先来看看被引用的段落：

In [9]:
cited_paragraphs = []
for paragraph in navigation_result["paragraphs"]:
    para_id = str(paragraph.get("display_id", str(paragraph["id"])))
    if para_id in answer.citations:
        cited_paragraphs.append(paragraph)
    

# Display the cited paragraphs for the audience
print("\n==== CITED PARAGRAPHS ====")
for i, paragraph in enumerate(cited_paragraphs):
    display_id = paragraph.get("display_id", str(paragraph["id"]))
    print(f"\nPARAGRAPH {i+1} (ID: {display_id}):")
    print("-" * 40)
    print(paragraph["text"])
    print("-" * 40)


==== CITED PARAGRAPHS ====

PARAGRAPH 1 (ID: 0.0.5.0):
----------------------------------------
104  Business to be Conducted in Writing
37 C.F.R. § 2.190(b)  Electronic trademark documents. … Documents that r elate to proceedings before
the Trademark Trial and Appeal Board must be filed electronically with the Board through ESTTA. 37 C.F.R. § 2.191 Action of the Office based on the written record. All business with the Office must be
transacted in writing. The action of the Office will be based exclusively on the written record. No consideration
will be given to any alleged oral promise, stipulation, or understanding when there is disagreement or doubt. With the exceptions of discovery conferences with Board participation, see TBMP § 401.01, and telephone
conferences, see TBMP § 413.01 and TBMP § 502.06, all business with the Board should be transacted in
writing. 37 C.F.R. § 2.191 . The personal attendance of parties or their attorne ys or other authorized
representatives at the off

“Literal 列表”技巧强制模型只能引用特定的段落 ID（如 "0.0.5.4"），而不会自己编造参考或高亮随机文本——可以把它想象成创建了一个数字化的“目录”，GPT-4.1 只能从中选择。这一方案确保你能获得可验证的、可追溯到确切源材料的引用链，解决了长上下文 RAG 中的一个重要问题。

最后，我们使用 LLM-as-judge 方法来验证答案。

In [10]:
from typing import List, Dict, Any, Literal
from pydantic import BaseModel

class VerificationResult(BaseModel):
    """Verification result format"""
    is_accurate: bool
    explanation: str
    confidence: Literal["high", "medium", "low"]

def verify_answer(question: str, answer: LegalAnswer, 
                 cited_paragraphs: List[Dict[str, Any]]) -> VerificationResult:
    """
    Verify if the answer is grounded in the cited paragraphs.
    
    Args:
        question: The user's question
        answer: The generated answer
        cited_paragraphs: Paragraphs cited in the answer
        
    Returns:
        Verification result with accuracy assessment, explanation, and confidence level
    """
    print("\n==== VERIFYING ANSWER ====")
    
    # Prepare context with the cited paragraphs
    context = ""
    for paragraph in cited_paragraphs:
        display_id = paragraph.get("display_id", str(paragraph["id"]))
        context += f"PARAGRAPH {display_id}:\n{paragraph['text']}\n\n"
    
    # Prepare system prompt
    system_prompt = """You are a fact-checker for legal information.
Your job is to verify if the provided answer:
1. Is factually accurate according to the source paragraphs
2. Uses citations correctly

Be critical and look for any factual errors or unsupported claims.
Assign a confidence level based on how directly the paragraphs answer the question:
- high: The answer is comprehensive, accurate, and directly supported by the paragraphs
- medium: The answer is mostly accurate but may be incomplete or have minor issues
- low: The answer has significant gaps, inaccuracies, or is poorly supported by the paragraphs
"""
    
    response = client.responses.parse(
        model="o4-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"""
QUESTION: {question}

ANSWER TO VERIFY:
{answer.answer}

CITATIONS USED: {', '.join(answer.citations)}

SOURCE PARAGRAPHS:
{context}

Is this answer accurate and properly supported by the source paragraphs?
Assign a confidence level (high, medium, or low) based on completeness and accuracy.
            """}
        ],
        text_format=VerificationResult
    )
    
    # Log and return the verification result
    print(f"\nAccuracy verification: {'PASSED' if response.output_parsed.is_accurate else 'FAILED'}")
    print(f"Confidence: {response.output_parsed.confidence}")
    print(f"Explanation: {response.output_parsed.explanation}")
    
    return response.output_parsed

# Verify the answer using only the cited paragraphs
verification = verify_answer(question, answer, cited_paragraphs)

# Display final result with verification
print("\n==== FINAL VERIFIED ANSWER ====")
print(f"Verification: {'PASSED' if verification.is_accurate else 'FAILED'} | Confidence: {verification.confidence}")
print("\nAnswer:")
print(answer.answer)
print("\nCitations:")
for citation in answer.citations:
    print(f"- {citation}")


==== VERIFYING ANSWER ====

Accuracy verification: PASSED
Confidence: high
Explanation: The answer correctly states that motions to compel discovery must be filed electronically through ESTTA, with paper submissions permitted only under the limited exceptions of technical failure or extraordinary circumstances (37 C.F.R. § 2.190(b) and 2.193(b)). It accurately describes the required title and caption placement (TBMP § 105), and it appropriately summarizes the signature requirements for electronic filings (37 C.F.R. § 2.193(c) and TBMP §§ 106.02, 106.02(b)–(e)), including the use of slash‐enclosed electronic signatures and identification of the signatory’s name and title. It also correctly notes the rule regarding defective signatures (37 C.F.R. § 2.119(e) and TBMP § 106.02). The citations align with the source paragraphs. 

==== FINAL VERIFIED ANSWER ====
Verification: PASSED | Confidence: high

Answer:
A motion to compel discovery must be filed electronically with the Trademark Trial

验证步骤会生成一份清晰、结构化的评估，引用具体法规，并有条不紊地检查答案的准确性及其对引用的正确使用。它不只是说“正确”，而是通过准确解释答案为什么正确来提供有用的上下文，让你有信心将附带具体引用的答案呈现给用户

## 4. 基础设施成本

我们来拆解这种智能体式 RAG 方法的成本结构：

### 估算的固定成本与可变成本

* **估算的固定（一次性）成本：**  
  * **传统 RAG：** 约 $0.43（嵌入 + 元数据生成）
  * **智能体式 RAG：** $0.00（无需任何预处理）


* **估算的可变（每次查询）成本：**  
  * **路由模型（`gpt-4.1-mini`）：**  
    * 初始路由（20 个块）：约 $0.10  
    * 两层递归：约 $0.20
  * **综合（`gpt-4.1`）：** 约 $0.05
  * **验证（`o4-mini`）：** 约 $0.01
  * **每次查询合计：** 约 $0.36

虽然每次查询的成本高于传统 RAG，但这种方法具有以下优势：
- 对新文档立即给出结果
- 引用更精确
- 更好地处理改述和概念性问题
- 无需承担基础设施维护开销

可以通过以下方式优化成本：
- 缓存常见查询的结果
- 限制模型调用中的最大 token 数
- 采用先对文档进行预过滤的混合方法

## 5. 与传统 RAG 相比的优势和权衡

### 优势
- **零摄取延迟**：无需预处理，即可立即回答关于新文档的问题。
- **动态导航**：通过聚焦于有希望的章节来模拟人类的阅读模式。
- **跨章节推理**：模型能够发现文档各章节之间的关联，而独立的块检索可能会遗漏这些关联，从而有望提高生成答案的准确性，并节省优化检索管道的时间。

### 权衡
- **每次查询成本更高**：与基于嵌入的检索相比，每个问题都需要更多计算。
- **延迟增加**：分层导航的处理时间比简单的向量查找更长。
- **可扩展性有限**：对于超大型文档集合，预处理反而更高效，这种方法可能难以应对。

## 6. 后续步骤

我们可以对所采用的方法做几项改进：
- **生成知识图谱**：我们可以利用 GPT 4.1-mini 的大上下文窗口迭代生成详细的知识图谱，然后由 GPT 4.1 遍历该图谱来回答问题。这样无论问什么问题，文档都只需要“摄取”一次。
- **改进草稿本工具**：可以为草稿本工具提供更多选项，例如编辑或删除过去的记忆。这将让模型能够选择与当前问题最相关的内容
- **调整深度**：我们可以调整分层导航的深度，在成本和性能之间找到合适的平衡。某些用例需要句子级引用（如法律文档），而另一些可能只需要段落级引用（如新闻文章）。 

## 7. 要点总结

1. **上下文窗口是一项超能力：** 百万 token 的上下文窗口使即时导航文档成为可能。
2. **分层方法模拟人类阅读：** 智能体式路由的工作方式就像人类略读文档以寻找相关章节。
3. **草稿本支持多步骤推理：** 维护推理记录可以提高导航质量。
4. **实现快速，无需数据库：** 整个系统只需 API 调用即可构建，无需基础设施。
5. **验证提高可靠性：** LLM-as-judge 模式能在错误到达用户之前将其捕获。

================================================================================

## 3B. 用例：面向制药研发的 AI 联合科学家
![面向制药研发的 AI 联合科学家](../../../images/3B_reasoning_task_card.png)

本节详细介绍如何构建一个充当“联合科学家”的 AI 系统，以加速制药研发中的实验设计，重点是在特定约束下优化药物合成流程。

## 🗂️ 太长不看矩阵

下表总结了这一特定 AI 联合科学家实现的核心技术选择及其理由。

| 层              | 选择                                                                  | 用途                                                                                           |
| :----------------- | :------------------------------------------------------------------------ | :------------------------------------------------------------------------------------------------------- |
| **构思**       | `o4-mini`（并行角色扮演智能体）                                  | 快速且经济高效地生成多样化的假设和方案；角色扮演可增强创造力。 |
| **事实依据**      | 外部工具调用（`chem_lookup`、`cost_estimator`、`outcome_db` 等） | 确保方案基于真实世界的数据（化学性质、成本、以往结果）。                   |
| **排序**        | `o4-mini`（两两锦标赛比较）                                | 超越简单打分的细致评估；高效选出有潜力的候选方案。                      |
| **评审/综合** | `o3`（深度评审与综合）                                            | 提供严谨的、资深专家级的分析，识别风险，并确保科学有效性。             |
| **安全（可选）**  | `gpt-4.1-mini`（针对性检查）                                       | 在移交给人工之前，增加一层专门的安全审查。                                   |
| **学习**       | `o3` + Code Interpreter（结果分析 → 数据库）                            | 系统地记录实验结果，实现随时间推移的持续改进。                |
| **核心技术** | 多智能体协作与升级                                    | 针对复杂的多步骤推理任务，发挥不同模型的优势（速度与深度）。      |

*注意：模型标识符截至 2025 年 4 月准确，可能会发生变化。*

## 1. 场景速览

* **问题领域：** 优化制药研发中复杂的实验流程，例如在遵守严格约束的前提下提高一种新药化合物（"XYZ-13"）的合成收率。
* **用户：** 参与药物发现和开发的研究科学家和实验室技术人员。
* **典型问题：**
    1. 提出 3 种不同的方案，通过测试不同的催化剂将 XYZ-13 的收率提高 ≥15%，使用已批准的试剂且费用控制在 $15k 以内。
    2. 提出在 60°C 以下（由于过去出现过过热问题）优化 XYZ-13 收率的方案，在预算范围内探索不同的已批准溶剂。
    3. 设计两种 XYZ-13 收率策略（目标 ≥15%）：a. 一种在 \$15k 预算内最大化潜在收率，b. 一种优先考虑成本、控制在 \$10k 以内。
* **约束：**
    * **预算：** 在规定的财务限额内运作（例如每个实验系列 $15,000）。
    * **法规/安全：** 只使用预先批准的化学品/试剂，并严格遵守安全规程。
    * **人工监督：** 最终的实验方案在执行前必须经过人类专家的审查和确认。

> 传统上，优化此类实验需要数周的人工规划、文献综述、反复的实验台工作和分析。这种 AI 联合科学家方法旨在通过自动化假设生成、方案设计和初步评估来大幅缩短周期，使科学家能够专注于更高层次的策略和最终验证。它将科学家的角色从手动执行规划步骤，转变为专家监督以及与 AI 协作。


## 2. 架构（多智能体推理）

该系统采用模拟高绩效科研团队的多智能体架构。不同的 AI 组件扮演专门的角色（例如构思、评审以及从结果中学习），并使用各种模型和工具协作执行工作流。

![AI 联合科学家架构](../../../images/3B_coscientist_architecture.png)

### 2.1. **科学家输入与约束：** 
流程从科学家定义目标、目标化合物和约束条件开始。

In [11]:
from openai import OpenAI
from agent_utils import Context, call_openai, log_json

# Example Initial Input
user_input = {
    "compound": "XYZ-13",
    "goal": "Improve synthesis yield by 15%",
    "budget": 15000,
    "time_h": 48,
    "previous": "Prior attempts failed at high temp; explore potential catalyst effects."
}
ctx = Context(client=OpenAI(), **user_input)

### 2.2.  **构思（`o4-mini` + 工具）：** 
多个 `o4-mini` 实例被赋予不同的角色提示（例如 `Hypothesis Agent`、`Protocol Agent`、`Resource Agent`），并行生成实验方案。分配不同的角色设定有助于激发多样化的视角，并在构思阶段同时覆盖问题的不同方面。

In [12]:
ROLE_FOCUS = {
    # Hypothesis Agent Prompt
    "hypothesis_agent": """You are a pharmaceutical hypothesis specialist. 
        Focus exclusively on analyzing the compound structure and research goals to generate testable hypotheses. 
        Consider mechanism of action, binding affinity predictions, and potential off-target effects.""",

    # Protocol Agent Prompt
    "protocol_agent"  : """You are a laboratory protocol specialist. 
        Design experimental procedures that will effectively test the provided hypothesis. 
        Focus on experimental conditions, controls, and measurement techniques.""",

    # Resource Agent Prompt
    "resource_agent"  : """You are a laboratory resource optimization specialist. 
        Review the proposed protocol and optimize for efficiency. 
        Identify opportunities to reduce reagent use, equipment time, and overall costs while maintaining scientific validity.""",
}

# Create a structured prompt template for ideation
IDEATION_PROMPT = """You are a pharmaceutical {role} specialist. Your goal is to {goal} for compound {compound}.
Constraints:
- Budget: ${budget}
- Approved reagents only
- Complete within {time_h} hours
- Previous attempts: {previous}
Respond with structured JSON describing your protocol."""

In [13]:
import json, logging
from pathlib import Path
from typing import Dict, List, Any, Optional
from dataclasses import asdict
from functools import partial

MODEL_IDEATE   = "o4-mini-2025-04-16"  # o4-mini model for ideation - balances speed and quality

# Configure logging to help with tracking experiment progress and debugging
logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.info(f"Run‑id {ctx.run_id}  Compound: {ctx.compound}")
logging.info(f"Logs will be stored in: {Path('logs') / ctx.run_id}")

def ideation(ctx: Context):
    logging.info("Starting ideation phase...")
    ideas = []
    for role, focus in ROLE_FOCUS.items():
        logging.info(f"Running ideation agent ${role}")
        sys = IDEATION_PROMPT.format(role=role, focus=focus, **ctx.prompt_vars())
        usr = f"Design a protocol to {ctx.goal} within ${ctx.budget}."
        idea = call_openai(ctx.client, MODEL_IDEATE, sys, usr, ctx)
        ideas.append(idea)
    log_json("ideation_done", ideas, ctx)
    return ideas

Run‑id 9835f69c  Compound: XYZ-13
Logs will be stored in: logs/9835f69c


构思智能体可以使用 `literature_search`、`chem_lookup`（化学数据库）、`cost_estimator`、`outcome_db`（以往实验的结果）等外部工具，使其建议有数据作为依据。明确启用并提示模型使用外部工具，可以确保生成的方案可行、合规，并参考了已有知识。模型会根据任务决定何时调用哪个工具。

In [14]:
IDEATION_PROMPT += """\nUse the following tools as appropriate:
- Use the `list_available_chemicals` tool to get list of approved reagents.
- Use the `chem_lookup` tool to verify properties of reagents mentioned.
- Use the `cost_estimator` tool to calculate the approximate cost based on reagents and proposed steps.
- Check the `outcome_db` for relevant prior experiments with {compound}"""

ideas = ideation(ctx)
logging.info("Ideation complete!")

Starting ideation phase...
Running ideation agent $hypothesis_agent
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
(Tool) List available chemicals
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
(Tool) Outcome DB: XYZ-13, yield, 5
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
(Tool) Cost estimator: [{'name': 'Palladium chloride', 'amount': 0.05, 'unit': 'g'}, {'name': 'Triphenylphosphine', 'amount': 0.1, 'unit': 'g'}, {'name': 'Potassium carbonate', 'amount': 1, 'unit': 'g'}, {'name': 'Dimethylformamide', 'amount': 50, 'unit': 'mL'}, {'name': 'Toluene', 'amount': 50, 'unit': 'mL'}, {'name': 'Sodium borohydride', 'amount': 0.1, 'unit': 'g'}, {'name': 'Triethylamine', 'amount': 0.5, 'unit': 'mL'}], ['round-bottom flask', 'magnetic stirrer', 'reflux condenser'], 36
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Running ideation agent $protocol_agent
HTTP Requ

这些工具定义在 `agent_utils.py` 中。在本方案中，工具调用在 `tools.py` 中进行了模拟。在实际用例中，这些工具会调用真实的 API。


### 2.3. **锦标赛排序（`o4-mini` / `o3`）：** 
生成的方案会根据预期效果、可行性、成本和新颖性等标准进行两两比较。与其让模型孤立地为方案打分，不如每次提供两个方案，并要求模型根据具体标准直接比较，这样通常能得到更可靠的相对排序。

这种 Elo 式排序能够找出最有潜力的候选方案，供进一步深入评审。

In [15]:
TOURNAMENT_PROMPT = """
Protocol A: [details...]
Protocol B: [details...]

Compare Protocol A and Protocol B for synthesizing {compound} aimed at {goal}. Score them on:
1. Likelihood of achieving ≥ 15% yield increase.
2. Practical feasibility (reagents, time).
3. Estimated cost-efficiency (use tool if needed).
4. Scientific novelty/risk.

Return JSON {{\"winner\": \"A\"|\"B\", \"justification\": \"...\"}}."""

# This is a mock tourname implementation that only compares the first two protocols
# A real implementation would compare pairs in a tournament bracket style
def tournament(protocols: List[Dict[str, Any]], ctx: Context):
    logging.info("Starting tournament phase...")
    if len(protocols) == 1:
        return protocols[:1]
    a, b = protocols[0], protocols[1]
    sys = TOURNAMENT_PROMPT.format(**ctx.prompt_vars())
    usr = json.dumps({"A": a, "B": b}, indent=2)
    res = call_openai(ctx.client, MODEL_IDEATE, sys, usr, ctx)
    winner = a if res.get("winner", "A").upper() == "A" else b
    log_json("tournament", res, ctx)
    return [winner]

top_proto = tournament(ideas, ctx)[0]
logging.info("Tournament winner picked!")

Starting tournament phase...
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Tournament winner picked!


> 在早期实验中，我们发现让模型按 1-10 分为方案打分会导致结果不一致，且分数趋于集中。锦标赛方法通过强制进行相对判断解决了这个问题，事实证明这种判断更可靠。这与人类专家的行为相似——科学家通常发现直接比较两个选项比给出绝对分数更容易。

### 2.4. **深度评审与综合（`o3`）：** 
排名靠前的方案会交给 `o3` 进行严格评审。`o3` 扮演资深科学家的角色，评估科学有效性、方法论、安全性和预算合规性，并提出改进建议或综合出最终的精炼方案。它也可能调用工具进行验证。

In [16]:
# Deep critique phase using a more powerful model for rigorous review
CRITIQUE_PROMPT = """You are a senior researcher reviewing a proposed synthesis protocol 
for {compound} aiming for {goal}, budget ${budget} using approved reagents. Review the protocol below rigorously:
1. Identify scientific flaws or methodological weaknesses.
2. Assess safety risks and budget compliance (use `cost_estimator` tool if needed).
3. Check for consistency with prior `outcome_db` results if relevant.
4. Suggest concrete improvements or rewrite sections if necessary.
5. Provide a final go/no-go recommendation.

Return JSON {{\"revised_protocol\": ..., \"critique\": \"...\", \"recommendation\": \"go|no-go\"}}.

Protocol to Review:
[Protocol details...]
"""

MODEL_CRITIQUE = "o3-2025-04-16"  # o3 model for deep critique

def critique(protocol: Dict[str, Any], ctx: Context):
    logging.info("Starting critique phase...")
    sys = CRITIQUE_PROMPT.format(**ctx.prompt_vars())
    usr = json.dumps(protocol, indent=2)
    crit = call_openai(ctx.client, MODEL_CRITIQUE, sys, usr, ctx)
    log_json("critique", crit, ctx)
    return crit.get("revised_protocol", protocol)

critiqued = critique(top_proto, ctx)
logging.info("Deep critique completed!")

Starting critique phase...
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
(Tool) Cost estimator: [{'name': 'Palladium chloride', 'amount': 0.0045, 'unit': 'g'}, {'name': 'Triphenylphosphine', 'amount': 0.013, 'unit': 'g'}, {'name': 'Sodium borohydride', 'amount': 0.0038, 'unit': 'g'}, {'name': 'Potassium carbonate', 'amount': 0.14, 'unit': 'g'}, {'name': 'Triethylamine', 'amount': 0.07, 'unit': 'mL'}, {'name': 'Dimethylformamide', 'amount': 2, 'unit': 'mL'}, {'name': 'Toluene', 'amount': 5, 'unit': 'mL'}], ['100 mL round-bottom flask', 'magnetic stirrer', 'reflux condenser', 'inert gas line'], 24
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
(Tool) Outcome DB: XYZ-13, None, 5
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Deep critique completed!


> 我们有意使用不同的模型和角色设定将构思与评审分开。让同一个模型既生成又评审自己的工作，往往会导致自我辩护，而不是客观评估。扮演“资深科学家”的 o3 模型始终能发现 o4-mini 在构思阶段遗漏的方法论缺陷。

### 2.5. **（可选）安全检查：** 
可以由一个专门的模型（例如 `gpt-4.1-mini`）针对特定的安全问题（例如危险的试剂组合）进行最终检查。

In [17]:
# Optional safety check using a targeted model
SAFETY_PROMPT = """You are a lab‑safety specialist. 
Identify hazards, unsafe conditions, or compliance issues in this protocol for {compound}. 
Use `chem_lookup` tool if needed. Return JSON assessment."""

MODEL_SAFETY   = "gpt-4.1-mini-2025-04-14"  # gpt-4.1-mini model for safety checks - optimized for instruction following

def safety(protocol: Dict[str, Any], ctx: Context):
    logging.info("Starting safety assessment...")
    sys = SAFETY_PROMPT.format(**ctx.prompt_vars())
    usr = json.dumps(protocol, indent=2)
    assessment = call_openai(ctx.client, MODEL_SAFETY, sys, usr, ctx)
    log_json("safety", assessment, ctx)
    return {"protocol": protocol, "safety": assessment}

secured = safety(critiqued, ctx)
logging.info("Safety check completed!")

Starting safety assessment...
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
(Tool) Chemical lookup: Palladium chloride, None
(Tool) Chemical lookup: Triphenylphosphine, None
(Tool) Chemical lookup: Sodium borohydride, None
(Tool) Chemical lookup: Potassium carbonate, None
(Tool) Chemical lookup: Dimethylformamide, None
(Tool) Chemical lookup: Toluene, None
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Safety check completed!


### 2.6. **人工审查：** 
AI 生成的最终方案通过界面呈现给人类科学家，供其确认、修改并最终批准。

In [18]:
def human_review(safety_package: Dict[str, Any], ctx: Context):
    logging.info("Awaiting human review...")
    protocol = safety_package["protocol"]
    safety_assessment = safety_package["safety"]
    
    print(f"\n=== PROTOCOL FOR REVIEW: {ctx.compound} - {ctx.goal} ===")
    print(f"DETAILS: {json.dumps(protocol, indent=2)}")
    print(f"SAFETY: {json.dumps(safety_assessment, indent=2)}")
    
    while True:
        approval = input("\nApprove for execution? (yes/no): ").lower()
        if approval in ['yes', 'y', 'no', 'n']:
            approved = approval in ['yes', 'y']
            logging.info(f"Protocol {'approved' if approved else 'rejected'}")
            return {"protocol": protocol, "approved": approved}
        print("Please enter 'yes' or 'no'")

human_decision = human_review(secured, ctx)

Awaiting human review...



=== PROTOCOL FOR REVIEW: XYZ-13 - Improve synthesis yield by 15% ===
DETAILS: {
  "protocol_title": "Optimised In-Situ Pd(0)/PPh3 Coupling for XYZ-13 \u2013 Target \u2265 72 % Yield",
  "key_changes_vs_original": [
    "Catalyst loading reduced from 5 mol % to 2 mol % Pd to cut cost and metal contamination without loss of activity.",
    "Reaction run at 0.10 M substrate concentration (12 mL solvent total) instead of 50 mL; higher effective collision frequency boosts conversion and reduces waste.",
    "Single solvent system (toluene/DMF 4:1) avoids phase separation and simplifies work-up.",
    "Redundant triethylamine removed; K2CO3 (2.5 eq) provides sufficient basicity.",
    "Reaction temperature raised slightly to 80 \u00b0C (still below side-reaction threshold found in exp-001) and time shortened to 24 h with in-process HPLC check at 6 h intervals.",
    "Work-up switched from large silica column to two-step: (a) aqueous EDTA wash to strip Pd, (b) recrystallisation from EtOAc/he

Protocol approved


### 2.7. **执行与学习（`o3` + Code Interpreter）：** 
人工批准后，方案会被送往实验室执行。实验室执行完成后，结果会反馈到系统中。`o3` 结合 `Code Interpreter` 分析数据、生成洞察，并将结构化的结果（方案、参数、结果、洞察）存储到数据库（`Outcome DB`）中。该数据库为未来的构思周期提供参考，形成一个学习闭环。

In [19]:
# Simulating execution and analyzing results
ANALYSIS_PROMPT = """You are a data analyst. 
Did the experiment achieve {goal}?  Analyse factors, suggest improvements, and return structured JSON.
"""

def execute_and_analyse(pkt: Dict[str, Any], ctx: Context):
    logging.info("Starting mock execution and analysis...")
    # These are mock results for a lab experiment
    mock_results = {
        "yield_improvement": 12.5,
        "success": False,
        "actual_cost": ctx.budget * 0.85,
        "notes": "Mock execution"
    }
    sys = ANALYSIS_PROMPT.format(**ctx.prompt_vars())
    usr = json.dumps({"protocol": pkt, "results": mock_results}, indent=2)
    analysis = call_openai(ctx.client, MODEL_CRITIQUE, sys, usr, ctx)
    log_json("analysis", analysis, ctx)
    return analysis

# Only proceed to execution if approved by the human reviewer
if human_decision["approved"]:
    summary = execute_and_analyse(human_decision, ctx)
    logging.info("Analysis complete")
else:
    logging.info("Protocol rejected by human reviewer - execution skipped")
    summary = None

Path("output").mkdir(exist_ok=True)
out_path = Path("output") / f"{ctx.run_id}_summary.json"
out_path.write_text(json.dumps(summary, indent=2))
print(f"\n🎉 Completed. Summary written to {out_path}")

Starting mock execution and analysis...
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
(Tool) Literature search: Pd(0) PPh3 coupling yield optimization EDTA work-up recrystallization losses, None, 3
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
(Tool) Outcome DB: XYZ-13, yield, 5
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Analysis complete



🎉 Completed. Summary written to output/9835f69c_summary.json


## 3. 模型使用手册

在 `o4-mini` 和 `o3` 之间如何选择，取决于任务的复杂度和所需的深度。对于其他任务，`gpt-4.1-mini` 在成本和性能之间取得了平衡；当需要更强的能力或更细致的处理时，推荐使用更强大的 `gpt4.1`。

| 任务               | 起步模型     | 何时升级...                                            | 升级到  | 理由                                                                                    |
| :----------------- | :------------- | :--------------------------------------------------------- | :----------- | :------------------------------------------------------------------------------------------- |
| 构思与方案生成 | `o4-mini` | 假设缺乏复杂化学合成所需的深度或创造力。 | `o3` | `o4-mini` 能快速、经济高效地生成多样化的方案。当需要更细致的方法时，`o3` 能提供更深入的科学推理。 |
| 方案排序 | `o4-mini` | 比较需要更深入的科学评估或多因素权衡。 | `o3` | 使用 `o4-mini` 进行锦标赛式排序能高效找出有潜力的候选方案。当需要评估细微的科学有效性时再升级。 |
| 深度评审与综合 | `o3` | 不适用——这项关键任务已在使用能力最强的模型。 | 不适用 | `o3` 擅长严谨的科学评审、识别方法论缺陷，以及综合复杂方案中的改进之处。这项任务本身就需要深度推理。 |
| 安全评估 | `gpt-4.1-mini` | 特定领域的危害需要更高的准确率或专门知识。 | `gpt-4.1` | 对于标准安全检查，`gpt-4.1-mini` 在成本和性能之间取得了良好平衡。当复杂的安全风险需要更高的准确率或更细致的推理时，升级到 `gpt4.1`。 |

**关键洞察：**
> 这个用例展示了一种强大的模式：使用更快、更便宜的模型（`o4-mini`）实现广度和初步筛选，然后升级到更强大的模型（`o3`）实现深度、关键评审和综合。这种分层方法在有效控制计算成本的同时，兼顾了创造力/速度与严谨性/准确性。与工具的集成对于让 AI 的推理建立在可验证的真实世界数据之上至关重要。

## 4. 部署说明

将 AI 联合科学家从原型过渡到实验室使用需要周密的规划。

* **成本控制：**
    * 实现可配置的“模式”（例如 `Fast`、`Standard`、`Thorough`），通过调整 `o4-mini` 构思智能体的数量、`o3` 评审的深度或可选检查的使用，在结果质量与成本、延迟之间取得平衡。
    * 按阶段（构思、排序、评审）和按工具调用跟踪 token 使用量，实现细粒度的成本监控。
* **可观测性：**
    * 记录每个步骤的输入、输出、模型选择、工具调用/响应、延迟和 token 数量。
    * 监控锦标赛排序的表现以及 `o3` 评审的影响（例如方案被大幅修改或否决的频率）。
    * 跟踪用户交互：哪些方案被人类科学家批准、修改或否决。
* **安全与合规：**
    * 实施多层安全机制：提示中的约束、基于工具的检查（例如通过 `chem_lookup` 检查试剂兼容性）、可选的专用模型检查（`gpt-4.1-mini`）、自动过滤器（例如针对已知的危险组合）以及强制性的人工审查。
    * 确保工具端点（例如内部数据库）满足安全要求。
* **推广策略：** 
    * 先对过去的实验进行回顾性分析，然后进入影子模式（AI 与人类规划人员并行提出方案），接着在严密监控下进行有限的实际用例试用，最后再更广泛地推广。


## 5. 要点总结

1. **模型搭配产生协同效应**：`o4-mini` 能快速覆盖更多范围；`o3` 带来精确性和深度。
2. **工具集成让推理立足于现实**：化学品成本和安全约束等真实世界的数据为决策提供依据。
3. **人类科学家仍处于核心地位**：该系统通过消除繁琐的工作来增强专家的能力，而不是取代他们。


## 6. 实用 Cookbook 与资源

以下精选资源可作为 AI 联合科学家系统设计与实现的补充：

- **[Orchestrating Agents: Routines and Handoffs](https://cookbook.openai.com/examples/orchestrating_agents)** 使用例程和交接来组织多智能体工作流，与构思→排序→评审的管道相关。

- **[GPT-4.1 提示指南](https://cookbook.openai.com/examples/gpt4-1_prompting_guide)** 高级提示、工具使用和任务分解，用于提高评审和安全审查的准确性。

- **[Structured Outputs for Multi-Agent Systems](https://cookbook.openai.com/examples/structured_outputs_multi_agent)** 通过 schema 校验强制输出一致的 JSON，以实现智能体之间的互操作。

- **[Agents - OpenAI API](https://platform.openai.com/docs/guides/agents)**  
  使用 OpenAI 工具构建多智能体系统的综合指南，涵盖编排、工具使用以及作为本系统架构基础的最佳实践。

================================================================================



## 3C. 用例：保险理赔处理

![](../../../images/3C_insurance_task_card.png)

许多企业都面临将手填表单数字化的任务。在本节中，我们将演示如何使用 OpenAI 对手填的保险表单进行数字化和校验。虽然这是保险行业的常见问题，但同样的技术也可以应用于其他各种行业和表单，例如税表、发票等。

## 🗂️ 太长不看矩阵

下表总结了这一面向保险用例的特定 OCR 实现的核心技术选择及其理由。

| 层 | 选择 | 用途 |
| :---- | :---- | :---- |
| JSON 输出 | 使用 Pydantic 的结构化输出 | 易于指定格式，比 `JSON mode` 更能遵循 schema |
| OCR 与视觉 | `gpt-4.1` | 强大的 OCR 和视觉能力，支持结构化输出 |
| 推理 | `o4-mini` | 价格实惠但能力出色的推理，支持函数调用 |
| 表单校验 | 自定义函数调用 | 可以与自定义或内部数据库交互 |

\*注意：价格和模型标识符截至 2025 年 4 月准确，可能会发生变化。

## 1\. 场景速览

* **用户：** 目标用户是需要从手写表单中摄取数据的保险服务和运营团队。  
* **典型需求：** 每种表单都有不同的结构要求，需要提取的字段也各不相同。  
* **约束：**  
  * **准确性：** 需要高准确率，以确保数据正确且完整。  
  * **不确定性：** 系统必须处理数据中的不确定性，例如缺失的数据、含糊的数据以及同一字段的不同格式。如果模型无法消除不确定性，系统需要一种请求人工审查的机制。  
  * **性能与成本：** 虽然系统延迟并不关键，但在控制成本的同时需要高准确率。我们的成本目标是每处理 1000 页不超过 $20。

## 2\. 架构

解决方案的高层基本架构如下所示。

![](../../../images/3C_insurance_architecture.png)

这项任务很复杂，需要多种模型能力，包括视觉、函数调用、推理和结构化输出。虽然 `o3` 能够同时完成所有这些工作，但我们在实验中发现，仅使用 `o4-mini` 不足以达到所需的性能。由于 `o3` 的相对成本较高，我们转而选择了两阶段方法。

1. 第一阶段使用 GPT 4.1 的视觉能力。这一阶段经过优化，以最高准确率提取文本，把不确定性留给推理阶段处理，并且不做任何页面上不可见的假设。通过在第一阶段进行 OCR，推理模型就无需直接处理图像——考虑到推理模型还必须执行所有其他任务，直接处理图像可能颇具挑战。  
     
2. 第二阶段利用 `o4-mini` 的推理能力。我们使用 `o4-mini` 校验 OCR 的准确性，并将数据提取为结构化格式。重要的是，我们希望 o4-mini 充当第二道质量关卡 \-- 如果在这一阶段 OCR 结果不完整，我们可以用 o4-mini 来完善并校验原始结果。

为了具体演示其工作原理，我们来看一张保险表单的示例图片。

![](../../../images/3C_insurance_form.png)

虽然表单本身相当简单，但其中存在缺失的数据和含糊的信息，传统的 OCR 系统很难正确填写。首先，注意邮政编码和县名被省略了。其次，用户的电子邮件地址含糊不清 \-- 它可能是 `jsmith1@gmail.com`，也可能是 `jsmithl@gmail.com`。在接下来的章节中，我们将介绍一个设计良好的解决方案如何处理这些歧义并返回正确的表单结果。

**环境设置与库代码：**

为了让示例代码更清晰，我们将环境设置（例如 `pip install` 命令）和库函数拆分到单独的代码块中。这样在解决方案的每个步骤中，就能更容易地只关注相关逻辑。

In [6]:
# Install Python requirements
%pip install -qU pydantic "openai>=1.76.0"

# All imports
import os
import json

from pydantic import BaseModel

# Create the OpenAI client
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "sk-dummykey"))

Note: you may need to restart the kernel to use updated packages.


In [7]:
def run_conversation_loop(
    client,
    messages,
    tools,
    tool_handlers,
    response_format,
    model,
):
    """Run the OpenAI response completion loop, handling function calls via tool_handlers until parsing final response."""
    summaries = []
    while True:
        print(
            f"Requesting completion from model '{model}' (messages={len(messages)})"
        )
        response = client.responses.parse(
            model=model,
            input=messages,
            tools=tools,
            text_format=response_format,
            reasoning={"summary": "auto"},
        )
        summaries.append(response.output[0].summary)

        if not response.output_parsed:
            print("Assistant requested tool calls, resolving ...")

            reasoning_msg, tool_call = response.output
            messages.append(reasoning_msg)
            messages.append({
                "id": tool_call.id,
                "call_id": tool_call.call_id,
                "type": tool_call.type,
                "name": tool_call.name,
                "arguments": tool_call.arguments,
            })

            if tool_call.name in tool_handlers:
                try:
                    args = json.loads(tool_call.arguments)
                except Exception as exc:
                    print(
                        "Failed to parse %s arguments: %s", tool_call.name, exc
                    )
                    args = {}
                result = tool_handlers[tool_call.name](**args)
                messages.append(
                    {
                        "type": "function_call_output",
                        "call_id": tool_call.call_id,
                        "output": str(result),
                    }
                )
                print(f"Tool call {tool_call.name} complete, result: {str(result)}")
            else:
                print("Unhandled function call: %s", tool_call.name)

        if response.output_parsed is not None:
            print("Received parsed result from model")
            return response, summaries

**流程说明：第 1 阶段**

1. **图像：** 将用户用智能手机拍摄的表单图像传给模型。OpenAI 的模型可以接受多种图像格式，但我们通常使用 PNG 格式，以保持文字清晰并减少伪影。在本例中，我们通过一个公开可访问的内容 URL 将图像传给模型。在生产环境中，你通常会以签名 URL 的形式传递托管在你自己云存储桶中的图像。  
     
2. **结构化输出 Schema：** 我们定义一个 Pydantic 模型来设定输出数据的结构。该模型包含需要从表单中提取的所有字段，以及每个字段的相应类型。我们的模型分为几个子组件，每个子组件本身都是一个 Pydantic 模型，并由父模型引用。

In [8]:
class PersonContact(BaseModel):
    name: str
    home_phone: str
    work_phone: str
    cell_phone: str
    email: str

class Address(BaseModel):
    street: str
    city: str
    state: str
    zip: str
    county: str

class DwellingDetails(BaseModel):
    coverage_a_limit: str
    companion_policy_expiration_date: str
    occupancy_of_dwelling: str
    type_of_policy: str
    unrepaired_structural_damage: bool
    construction_type: str
    roof_type: str
    foundation_type: str
    has_post_and_pier_or_post_and_beam_foundation: bool
    cripple_walls: bool
    number_of_stories: str
    living_space_over_garage: bool
    number_of_chimneys: str
    square_footage: str
    year_of_construction: str
    anchored_to_foundation: bool
    water_heater_secured: bool

class InsuranceFormData(BaseModel):
    applicant: PersonContact
    co_applicant: PersonContact
    risk_address: Address
    mailing_address_if_different_than_risk_address: Address
    participating_insurer: str
    companion_policy_number: str
    dwelling_details: DwellingDetails
    effective_date: str
    expiration_date: str

3. **运行 OCR：** 利用 GPT-4.1 的视觉能力，我们运行管道的第一阶段，以结构化格式从文档中提取文本。这一初始阶段旨在实现高准确率，同时将不确定性传递给第二阶段。我们的提示明确指示模型不要推断输入，而是尽可能精确地填写细节。对于图像输入，我们将图像输入细节级别设置为 `auto`，以推断适合该图像的细节级别。我们在实验中发现 `auto` 效果很好，但如果你在 OCR 处理中遇到质量问题，可以考虑使用 `high`。

In [9]:
OCR_PROMPT = """You are a helpful assistant who excels at processing insurance forms.

You will be given an image of a hand-filled insurance form. Your job is to OCR the data into the given structured format.
Fill out the fields as exactly as possible. If a written character could possibly be ambiguous (i.e. l or 1, o or 0), include all possiblities in the field separated by "OR", especially for email addresses.
"""

user_content = [
    {"type": "input_text", "text": "Here is a photo of the form filled out by the user:"},
    {
        "type": "input_image",
        "image_url": "https://drive.usercontent.google.com/download?id=1-tZ526AW3mX1qthvgi8spaaxxeqFG5_6",
        "detail": "auto",
    },
]

messages = [
    {"role": "system", "content": OCR_PROMPT},
    {"role": "user", "content": user_content},
]

response = client.responses.parse(
    model="gpt-4.1-2025-04-14",
    input=messages,
    text_format=InsuranceFormData,
    # Set temp to 0 for reproducibility
    temperature=0,
)

s1_json_results = json.dumps(json.loads(response.output_parsed.model_dump_json()), indent=2)
print(s1_json_results)

{
  "applicant": {
    "name": "Smith, James L",
    "home_phone": "510 331 5555",
    "work_phone": "",
    "cell_phone": "510 212 5555",
    "email": "jsmithl@gmail.com OR jsmith1@gmail.com"
  },
  "co_applicant": {
    "name": "Roberts, Jesse T",
    "home_phone": "510 331 5555",
    "work_phone": "415 626 5555",
    "cell_phone": "",
    "email": "jrobertsjr@gmail.com"
  },
  "risk_address": {
    "street": "855 Brannan St",
    "city": "San Francisco",
    "state": "CA",
    "zip": "",
    "county": ""
  },
  "mailing_address_if_different_than_risk_address": {
    "street": "",
    "city": "",
    "state": "",
    "zip": "",
    "county": ""
  },
  "participating_insurer": "Acme Insurance Co",
  "companion_policy_number": "81265919",
  "dwelling_details": {
    "coverage_a_limit": "$900,000",
    "companion_policy_expiration_date": "5/31/27",
    "occupancy_of_dwelling": "Owner",
    "type_of_policy": "Homeowners",
    "unrepaired_structural_damage": false,
    "construction_type"

注意，输出中缺少几个字段。在下一个处理阶段，我们将利用 OpenAI 的推理模型尽可能推断出缺失的字段。

**流程说明：第 2 阶段**

1. **函数定义：** 我们定义了一组自定义函数，供模型用来消除不确定性。在本例中，我们定义了一个函数，通过检查电子邮件是否存在来校验电子邮件地址。这可用于解决含糊的电子邮件地址字段，即模型必须在多个可能的值之间做出选择的情况。默认情况下，o4-mini 支持网页搜索等内置工具，在本例中，它将使用网页搜索来确定邮政编码并补全不完整的地址。

In [10]:
tools = [{
    "type": "function",
    "name": "validate_email",
    "description": "Check if an email address is valid and exists.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address to validate."
            }
        },
        "required": [
            "email"
        ],
        "additionalProperties": False
    }
},
{
    "type": "function",
    "name": "search_web",
    "description": "Perform a web search.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to run through the search engine."
            }
        },
        "required": [
            "query"
        ],
        "additionalProperties": False
    }
}]

2. **提示：** 我们向模型提供一个提示，说明我们已经通过 OCR 提取了文本，并要求模型进行推理和函数调用，以填写缺失或含糊的字段。

In [11]:
PROMPT = """You are a helpful assistant who excels at processing insurance forms.

You will be given a javascript representation of an OCR'd document. Consider at which fields are ambiguous reason about how to fill them in. Fill any missing fields that are possible to infer from existing data, or search the web. If you cannot fill a field, reason about why.

Use the tools provided if necessary to clarify the results. If the OCR system has provided two possibilities, do your best to definitely pick which option is correct.
"""

In [12]:
messages = [
    {"role": "system", "content": PROMPT},
    {"role": "user", "content": s1_json_results},
]

# For demonstration purposes, we'll hardcode the correct email answer.
def email_mock(*args, **kwargs):
    if kwargs["email"] == "jsmithl@gmail.com":
        return True
    return False

# Reasoning models like `o4-mini` will soon support built-in web search, but for now
# we demonstrate this capability using a simple mock function.
def web_mock(*args, **kwargs):
    if "855 Brannan" in kwargs["query"]:
        return "855 Brannan St, San Francisco, 94103, San Francisco County"
    
    return ""
    
tool_handlers = {"validate_email": email_mock, "search_web": web_mock}

response, summaries = run_conversation_loop(
    client=client,
    messages=messages,
    tools=tools,
    tool_handlers=tool_handlers,
    response_format=InsuranceFormData,
    model="o4-mini-2025-04-16",
)

print(json.dumps(json.loads(response.output_parsed.model_dump_json()), indent=2))

Requesting completion from model 'o4-mini-2025-04-16' (messages=2)
Assistant requested tool calls, resolving ...
Tool call validate_email complete, result: True
Requesting completion from model 'o4-mini-2025-04-16' (messages=5)
Assistant requested tool calls, resolving ...
Tool call validate_email complete, result: False
Requesting completion from model 'o4-mini-2025-04-16' (messages=8)
Received parsed result from model
{
  "applicant": {
    "name": "Smith, James L",
    "home_phone": "510 331 5555",
    "work_phone": "",
    "cell_phone": "510 212 5555",
    "email": "jsmithl@gmail.com"
  },
  "co_applicant": {
    "name": "Roberts, Jesse T",
    "home_phone": "510 331 5555",
    "work_phone": "415 626 5555",
    "cell_phone": "",
    "email": "jrobertsjr@gmail.com"
  },
  "risk_address": {
    "street": "855 Brannan St",
    "city": "San Francisco",
    "state": "CA",
    "zip": "94107",
    "county": "San Francisco"
  },
  "mailing_address_if_different_than_risk_address": {
    "st

可以看到，电子邮件地址已被确定为单一值，邮政编码和县名已被填写，邮寄地址也已使用风险地址填写。模型还以结构化格式返回了结果（并使用了合适的类型，例如对是/否问题使用布尔值），下游系统可以很容易地对其进行解析。

为了帮助我们理解和调试模型，我们还可以打印模型生成的思维链推理摘要。这有助于暴露常见的失败模式、模型不确定的地方，或上游不正确的细节。

在开发这个解决方案时，思维链摘要就暴露出了一些命名和类型不正确的 schema 值。

In [13]:
for summary in summaries:
    for response in summary:
        print(response.text + '\n')

**Determining insurance form details**

I have a JSON representation of a partially filled insurance form, and there are a few missing or ambiguous fields that I need to address.

For the email address, I see two options. I can validate which one is correct by checking both with the tool.

The risk address fields for zip code and county are empty. Based on the address "855 Brannan St, San Francisco, CA," I can determine the correct zip code is 94107, as that area corresponds to South Beach. Lastly, since the mailing address is empty, I assume it's the same as the risk address.

**Filling insurance form details**

I think it’s best to set the mailing address to be the same as the risk address or clarify that a blank one implies the same. Since it’s an explicit instruction to fill missing fields, I’ll fill in the mailing address with the risk address to avoid confusion.

All co-applicant fields are present, and dwelling details are complete. The effective and expiration dates are also pr

## 3\. 模型与能力使用手册

为任务选择合适的工具是获得最佳结果的关键。一般来说，最好先从满足需求的最简单方案开始，在需要更多能力时再升级。

| 任务 | 起步模型 | 何时升级... | 升级到 | 理由 |
| :---- | :---- | :---- | :---- | :---- |
| OCR | `gpt-4.1` | 难以一眼看懂的复杂表单 | `o3` | 对于大多数 OCR 任务，`gpt-4.1` 快速且经济高效。`o-3` 能够对表单结构进行推理。 |
| 结果完善 | `o4-mini` | 推断细节的逻辑复杂，需要大量函数调用。 | `o3` | 更适合非常长的推理链，尤其是同时涉及函数调用和结构化输出时。 |

## 4\. 评估指标

跟踪关键指标，确保系统准确且按预期运行。

### 关键指标

* **OCR 准确率：** 按字符和按单词计算的准确率。  
* **推断字段比率：** 未填写的条目中，通过现有数据或函数调用被正确推断出来的比例。  
* **人工干预率：** 文档中包含 UNKNOWN、必须转交人工处理的频率。

我们建议构建一个带标注的保留表单集及其预期响应。该数据集应能代表预期的部署环境。有关构建和评估系统的更多详细信息，请参阅 [OpenAI evals](https://platform.openai.com/docs/guides/evals) 指南。

## 5\. 部署说明

从原型转向可用于生产的系统，需要关注运维细节（LLMOps）。

### 成本明细

我们假设对于文档摄取，由于对延迟的容忍度较高（例如通宵运行也没问题），[批处理定价](https://platform.openai.com/docs/guides/batch)是一个可行的选择。

#### **第 1 阶段：OCR（光学字符识别）**

**模型：** `gpt-4.1`

| 类型 | Token 数 | 费率（每 100 万） | 成本 |
| :---- | :---- | :---- | :---- |
| 输入 | 2,000 | $1.00 | $0.002 |
| 输出 | 1,500 | $4.00 | $0.006 |
| **1,000 页合计（第 1 阶段\）** |  |  | **$8.00** |

#### **第 2 阶段：推理**

**模型：** `o4-mini`

| 类型 | Token 数 | 费率（每 100 万） | 成本 |
| :---- | :---- | :---- | :---- |
| 输入 | 2,000 | $0.55 | $0.0011 |
| 输出 | 3,000 | $2.20 | $0.0066 |
| **1,000 页合计（第 2 阶段\）** |  |  | **$7.70** |

#### 总计（每 1,000 页）：**$15.70**

将这一成本与单阶段 `o3` 部署进行比较。假设 token 使用量和批处理使用方式相同，使用更强大的推理模型的额外成本将达到每 1000 页 $70。

### 监控与部署

通过记录关键指标来监控你的系统：

* 每个模型的 `llm_model_used`、`llm_input_tokens`、`llm_output_tokens`、`llm_latency_ms`  
* 每个模型的 `total_query_latency_ms`、`estimated_query_cost`  
* `function_calls_per_document`、`num_email_validation_calls`  
* `human_review_required`

通过配置/环境变量固定部署中使用的具体模型版本标识符（例如 `o4-mini-2025-04-16`），以防止模型静默更新带来意外行为。

## 6\. 实用 Cookbook 与资源

如需深入了解具体组件，请参阅以下相关资源：

* [Structured Output](https://platform.openai.com/docs/guides/structured-outputs)  
* [视觉模型](https://platform.openai.com/docs/guides/images)  
* [函数调用](https://platform.openai.com/docs/guides/function-calling)

================================================================================


<h2 id="prototype-to-production">从原型到生产</h2>

将原型过渡到生产需要周密的规划和执行。本检查清单借鉴了我们的旗舰用例，重点列出了关键步骤，以确保你的部署稳健、高效并满足业务目标。

## 🗂️ 太长不看矩阵

| 检查领域 | 重点 / 行动 | 为何重要 |
| :---- | :---- | :---- |
| **定义成功标准** | • 定义可衡量的 KPI 和 SLO（准确率、成本、延迟）。 • 确保目标可以通过日志衡量。 | 提供明确的目标；证明价值。 |
| **记录模型选择依据** | • 基于权衡审慎选择初始模型。 • 记录模型选择背后的“原因”。 | 为选择提供依据；便于未来更新。 |
| **稳健的评估与测试** | • 使用黄金数据集构建自动化测试（“评估套件”）。 • 重点关注事实性、幻觉、工具错误。  • 测试工具可靠性和边界情况。 | 确保质量；在发布前防止退化。 |
| **可观测性与成本** | • 实现用于监控和调试的必要日志记录。 • 设置成本护栏（token 上限、使用模式）。 | 便于调优；将支出控制在预算之内。 |
| **安全与合规** | • 使用安全机制（审核 API、提示）。 • 执行特定领域的合规规则。 • 对高风险输出强制要求人在回路（HITL）。 | 确保负责任地运行；满足要求。 |
| **模型更新与版本管理** | • 定义版本固定策略 • 为新版本实施 A/B 测试 • 制定回滚流程 | 在保持稳定的同时允许改进。 |

1. **量化定义成功标准：** 在大规模开发*之前*，从“能用就行”转向可衡量的目标。  
     
   * **设定关键绩效指标（KPI）和 SLO：** 为业务价值（例如 RAG 准确率 \> 95%、OCR 成本 \< $X/页）和性能（例如 P95 延迟 \< 1s、错误率）定义具体目标。  
   * **确保可衡量：** 确认所有 KPI 和 SLO 都可以直接从系统日志中衡量（例如跟踪 `total_tokens`、`critique_status`）。

   

2. **记录初始模型选择的依据：** 为你最初的模型选择提供依据，以备将来参考。  
     
   * **审慎选择模型：** 利用模型简介矩阵和用例为每项任务选择合适的模型（例如用 `o4-mini` 兼顾速度/成本，用 `gpt-4.1` 追求准确率，用 `o3` 追求深度）。  
   * **记录“原因”：** 在代码注释或设计文档中简要记录你做出选择的理由（成本、延迟、能力之间的权衡），让未来的团队了解背景。

   

3. **实施稳健的评估与测试：** 在发布变更*之前*验证质量并防止退化。  
     
   * **构建自动化评估套件：** 使用“黄金数据集”（50-100 个多样化、经专家验证的示例）创建可重复的测试流程。测试重点关注 `factuality`、`hallucination rate`、`tool-error rate` 以及特定任务的指标。  
   * **可靠地测试：** 严格测试集成工具的可靠性（成功率、错误处理），以及系统在负载下和边界情况（格式错误的数据、对抗性输入）下的行为。

   

4. **建立可观测性和成本控制：** 监控性能并将支出控制在预算之内。  
     
   * **设置成本护栏：** 通过为每个阶段定义最大 token 上限，并考虑采用运行模式（"Fast"、"Standard"、"Thorough"）来平衡成本和性能，从而防止成本意外上涨。  
   * **实现必要的日志记录：** 通过结构化日志记录每个处理阶段的关键运行数据，以便进行调试和监控。

   

5. **实施安全与合规护栏：** 确保负责任地运行并满足要求。  
     
   * **使用安全机制：** 使用 OpenAI 的审核 API、注重安全的系统提示或哨兵模型等工具进行检查，尤其是在处理用户输入或敏感话题时。  
   * **执行合规要求：** 内置与你所在行业和风险相关的检查（例如法律约束、实验室安全）。  
   * **要求人在回路（HITL）：** 对低置信度输出、高风险场景或关键决策强制进行人工审查，并确保工作流清楚地标记出这些条目。


6. **管理模型更新和版本：** 为模型随时间的演进做好准备。
   
   * **版本固定策略：** 决定是固定到特定模型版本以保持稳定，还是自动采用新版本以获得改进。
   * **A/B 测试框架：** 建立一套流程，在全面部署之前根据关键指标评估新的模型版本。
   * **回滚计划：** 制定清晰的流程，以便在更新出现问题时回退到之前的模型版本。
   * **监控版本表现：** 跟踪不同模型版本的指标，以识别性能趋势并为未来的选择决策提供依据。

================================================================================



## 适配决策树

![模型选择决策树](../../../images/3D_model_selection_flowchart.png)

## 向非技术利益相关者解释模型选择

在向业务利益相关者解释你的模型选择时，请重点关注以下几点：

1. **与业务成果保持一致**：说明你的模型选择如何直接支持具体的业务目标（节省时间、降低成本、提高准确率）。

2. **转化技术指标**：将技术考量转化为业务影响：
   - “该模型将处理时间从 5 秒缩短到 0.7 秒，使我们处理客户咨询的速度提高了 7 倍”
   - “通过使用 mini 变体，我们可以在相同预算内处理 5 倍的文档”

3. **突出权衡**：为不同模型呈现清晰的方案：
   - “方案 A（GPT-4.1）：准确率最高但成本较高——非常适合面向客户的法律分析”
   - “方案 B（GPT-4.1 mini）：以 30% 的成本达到 90% 的准确率——非常适合内部文档处理”

4. **使用具体示例**：展示不同模型在输出上的实际差异，以说明每个方案的价值主张。

================================================================================



## 附录

## 关键术语表

| 术语 | 定义 |
|------|------------|
| **上下文窗口（Context Window）** | 模型在单次请求中可以处理的最大 token 数 |
| **幻觉（Hallucination）** | 模型生成看似合理、但实际上不正确或缺乏依据的内容 |
| **延迟（Latency）** | 从向模型发送请求到收到响应之间的时间延迟 |
| **LLM** | 大语言模型（Large Language Model）；在海量文本数据上训练的 AI 系统 |
| **提示工程（Prompt Engineering）** | 设计有效提示以从 AI 模型获得期望输出的实践 |
| **RAG** | 检索增强生成（Retrieval-Augmented Generation）；将信息检索与文本生成相结合 |
| **SOTA** | 当前最佳水平（State-of-the-Art）；代表某一领域在特定时间的最先进阶段 |
| **Token** | 模型处理文本的基本单位（在英语中大约相当于 0.75 个单词） |

## 6.1 价格与用途表（2025 年 4 月）

| 模型 | 上下文窗口 | 输入价格（每 100 万 token） | 输出价格（每 100 万 token） | 最适用于 |
|-------|----------------|-----------------------------|-----------------------------|----------|
| GPT-4.1 | 1M | \$2.00 | \$8.00 | 长文档分析、代码审查 |
| GPT-4.1 mini | 1M | \$0.40 | \$1.60 | 生产环境智能体，兼顾成本/性能 |
| GPT-4.1 nano | 1M | \$0.10 | \$0.40 | 高吞吐量、对成本敏感的应用 |
| GPT-4o | 128K | \$5.00 | \$15.00 | 实时语音/视觉对话 |
| GPT-4o mini | 128K | \$0.15 | \$0.60 | 视觉任务、快速分析 |
| o3 (low) | 200K | \$10.00* | \$40.00* | 批量分诊、商品目录信息丰富 |
| o3 (med) | 200K | \$10.00* | \$40.00* | 知识库问答 |
| o3 (high) | 200K | \$10.00* | \$40.00* | 多步骤推理、故障排查 |
| o4-mini (low) | 200K | \$1.10* | \$4.40* | 视觉任务、快速分析 |
| o4-mini (med) | 200K | \$1.10* | \$4.40* | 视觉 + 推理兼顾 |
| o4-mini (high) | 200K | \$1.10* | \$4.40* | 可控成本下的深度推理 |

\* *注意：low/med/high 设置影响的是 token 使用量，而不是基础定价。更高的设置可能会为更深入的推理使用更多 token，从而增加每次请求的成本和延迟。*

## 6.2 提示模式速查表（token 与延迟变化）

| 提示模式 | 说明 | Token 影响 | 延迟影响 | 最适合的模型 |
|----------------|-------------|--------------|----------------|----------------|
| **自我评审** | 让模型在定稿前评估自己的答案 | token +20-30% | 延迟 +15-25% | GPT-4.1, o3 |
| **思维链（CoT）** | 明确指示“逐步思考” | token +40-80% | 延迟 +30-50% | o3, o4-mini (high) |
| **结构化输出** | 使用 JSON schema 或 pydantic 模型保证格式一致 | token +5-10% | 延迟 +5-10% | 所有模型 |
| **零 token 记忆** | 将上下文存储在外部数据库中，而不是放在对话里 | token -70-90% | 延迟 -5-10% | GPT-4.1 家族 |
| **骨架填充** | 提供模板结构供模型补全 | token -10-20% | 延迟 -5-15% | o4-mini, GPT-4.1 nano |
| **自洽性** | 生成多个答案并选择最一致的一个 | token +200-300% | 延迟 +150-250% | o3 (high) |
| **角色扮演** | 为模型分配特定角色以获取专门知识 | token +5-15% | 中性 | GPT-4o, o4-mini |
| **锦标赛排序** | 两两比较选项，而不是逐个打分 | token +50-100% | 延迟 +30-60% | o3, o4-mini (high) |
| **工具调用反射** | 提示模型在检测到不确定性时调用工具 | token +10-30% | 延迟 +20-40% | o3, GPT-4.1 |

## 6.3 外部 Cookbook 与文档链接

### OpenAI 官方资源
- [OpenAI Cookbook 主仓库](https://cookbook.openai.com/)
- [函数调用指南](https://platform.openai.com/docs/guides/function-calling)
- [视觉模型指南](https://platform.openai.com/docs/guides/vision)
- [Agents 文档](https://platform.openai.com/docs/guides/agents)
- [Structured Outputs 指南](https://platform.openai.com/docs/guides/structured-outputs)

### RAG 与检索
- [基于 PDF 的 RAG](https://cookbook.openai.com/examples/file_search_responses)

### 专门用例
- [使用 Agents SDK 构建语音助手](https://cookbook.openai.com/examples/agents_sdk/app_assistant_voice_agents)
- [多工具编排](https://cookbook.openai.com/examples/responses_api/responses_api_tool_orchestration)
- [数据提取与转换](https://cookbook.openai.com/examples/data_extraction_transformation)

### 提示与模型选择
- [GPT-4.1 提示指南](https://cookbook.openai.com/examples/gpt4-1_prompting_guide)
- [提示工程最佳实践](https://platform.openai.com/docs/guides/prompt-engineering)

### 评估与部署
- [OpenAI Evals 入门](https://cookbook.openai.com/examples/evaluation/getting_started_with_openai_evals)
- [如何使用 Usage API 和 Cost API 监控你的 OpenAI 使用情况](https://cookbook.openai.com/examples/completions_usage_api)

================================================================================



## 贡献者

 本 cookbook 由 OpenAI 与 [Tribe AI](https://www.tribe.ai/) 共同协作完成
- [Kashyap Coimbatore Murali](https://www.linkedin.com/in/kashyap-murali/)
- [Nate Harada](https://www.linkedin.com/in/nate-harada/) 
- [Sai Prashanth Soundararaj](https://www.linkedin.com/in/saiprashanths/)
- [Shikhar Kwatra](https://www.linkedin.com/in/shikharkwatra/)